In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")

device: cuda:0


### Hyperparameter, Augmentasi Ekstra & Data Loading

In [12]:
BATCH_SIZE = 16  
EPOCHS = 15      
LEARNING_RATE = 1e-3
IMG_SIZE = 300   

data_dir = '../Data/train_cropped/' 

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=val_transform)
class_names = full_dataset.classes

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_dataset.dataset.transform = train_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Data siap! Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Data siap! Train: 1152, Val: 289


### Inisialisasi Model B3, Scheduler, dan AMP Scaler

In [13]:
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=len(class_names), drop_rate=0.3)
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

### Advanced Training Loop dengan Early Stopping

In [15]:
train_losses, val_losses, val_accuracies = [], [], []
best_val_acc = 0.0

patience = 3 
patience_counter = 0

model_dir = '../Models/' 
os.makedirs(model_dir, exist_ok=True)

model_save_path = os.path.join(model_dir, 'efficientnet_b3_model.pth')

print("Training...")
for epoch in range(EPOCHS):
    
    model.train()
    running_loss = 0.0
    
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        
        current_lr = scheduler.get_last_lr()[0]
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{current_lr:.6f}"})
        
    scheduler.step()
    
    epoch_train_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]')
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct / total
    
    val_losses.append(epoch_val_loss)
    val_accuracies.append(epoch_val_acc)
    
    print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        
        torch.save(model.state_dict(), model_save_path)
        
        print(f"Model terbaik baru disimpan di {model_save_path}! (Akurasi: {best_val_acc:.4f})")
        patience_counter = 0 
    else:
        patience_counter += 1
        print(f"Tidak ada perbaikan. Early stopping counter: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(f"\nEarly Stopping diaktifkan! Proses training dihentikan pada Epoch {epoch+1} untuk mencegah overfitting.")
        break

print(f"\nTraining Selesai! Model paling stabil memiliki akurasi: {best_val_acc:.4f}")

Training...


Epoch 1/15 [Train]:   0%|          | 0/72 [00:00<?, ?it/s]

Epoch 1/15 [Val]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.5216 | Val Loss: 0.4823 | Val Acc: 0.8374
Model terbaik baru disimpan di ../Models/efficientnet_b3_model.pth! (Akurasi: 0.8374)


Epoch 2/15 [Train]:   0%|          | 0/72 [00:00<?, ?it/s]

...

Epoch 9/15 [Val]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 9 | Train Loss: 0.0418 | Val Loss: 0.5343 | Val Acc: 0.8893
Tidak ada perbaikan. Early stopping counter: 2/3


Epoch 10/15 [Train]:   0%|          | 0/72 [00:00<?, ?it/s]

Epoch 10/15 [Val]:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 10 | Train Loss: 0.0118 | Val Loss: 0.5844 | Val Acc: 0.9031
Tidak ada perbaikan. Early stopping counter: 3/3

Early Stopping diaktifkan! Proses training dihentikan pada Epoch 10 untuk mencegah overfitting.

Training Selesai! Model paling stabil memiliki akurasi: 0.9308
